# Embedding
A small version of FastText is used (100 dimensions) in order to make the embedding faster. The pipeline is the subsequent:
- An embedding model is trained on each snapshot (as defined in ```\understanding```). 
- Stopwords and words with < 3 characters are removed since they're considered noise, and the remaining words are transformed into vectors.

In [ ]:
from pathlib import Path
import fasttext
from collections import Counter
import numpy as np
import nltk
from nltk.corpus import stopwords
import os
import multiprocessing
from sklearn.metrics.pairwise import cosine_similarit
import random
import matplotlib.pyplot as plt
import sys

1. The models are trained on the different snapshots.

In [ ]:
# Model creation

for file in os.listdir("../lemmas/snaps"):
    i = 1
    input_file = Path(f"../lemmas/snaps/{file}")  
    model_out = Path(f"fasttext_snap{i}_{file}_lite")   # prefisso modello output

    # TRAINING FASTTEXT LEGGERO


    print("Avvio training FastText...")

    model = fasttext.train_unsupervised(
        input=str(input_file),
        model="skipgram",
        dim=100,
        minn=3,
        maxn=5,
        wordNgrams=1,
        # minCount=5,
        epoch=5,
        bucket=2000000,
        thread=multiprocessing.cpu_count()
    )

    # salva modello binario pronto per embedding
    model.save_model(f"{model_out}.bin")
    print(f"Modello binario salvato come {model_out}.bin")


2. Stopwords are removed with **nltk**, as well as words shorter than 3 characters (considered noise). Moreover, frequencies of all the words are computed for each snapshot and those with a frequency slower than the 10th-percentile are removed. Finally, word embeddings are generated. The resulting files are

- ```vocab_freq_snap{n}.csv``` = words (no stopwords and characters < 3) associated with their frequency. Less frequent words are present here.
- ```fasttext_snap{n}_filt.vec``` = word embeddings of words with frquency > threshold (10th-percentile).


In [ ]:
import fasttext
import numpy as np
from pathlib import Path
from collections import Counter
import nltk
from nltk.corpus import stopwords
import sys

# --- 1. SETUP GLOBALE (fatto una volta sola) ---
try:
    nltk.data.find('corpora/stopwords')
except LookupError:
    nltk.download('stopwords')

stopwords_it = set(stopwords.words('italian'))

def stream_tokens(file_path, valid_vocab_set):
    """
    Generatore che legge il file riga per riga e restituisce i token validi.
    Non carica mai tutto il file in memoria.
    """
    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            # Tokenizzazione base (split su spazi)
            for token in line.lower().split():
                # --- FILTRI IN ORDINE DI VELOCITÀ ---
                
                # 1. Lunghezza (il controllo più veloce)
                if len(token) < 3:
                    continue
                
                # 2. Numeri (se contiene cifre, scartiamo)
                if any(char.isdigit() for char in token):
                    continue
                
                # 3. Stopwords (lookup in set O(1))
                if token in stopwords_it:
                    continue
                
                # 4. Vocabolario del modello (filtro finale)
                # Contiamo solo se la parola esiste nel modello FastText caricato
                if token in valid_vocab_set:
                    yield token

# --- 2. CICLO DI ANALISI ---
for n in range(1, 11):
    print(f"\n--- Processing Snap {n} ---")
    
    # A. Caricamento Modello
    model_path = f"fasttext_snap{n}_lite.bin"
    if not Path(model_path).exists():
        print(f"Modello {model_path} non trovato, salto.")
        continue
        
    print(f"Caricamento modello {n}...", end="")
    model = fasttext.load_model(model_path)
    
    # Estraiamo il vocabolario come SET per lookup istantaneo O(1)
    vocab_set = set(model.get_words())
    print(f" Fatto. Vocab size: {len(vocab_set)}")

    # B. Calcolo Frequenze (Streaming)
    input_file = Path(f"../lemmas/snaps/snap{n}.txt")
    
    print("Calcolo frequenze (streaming)...", end="")
    # Counter consuma il generatore senza caricare liste in RAM
    freq = Counter(stream_tokens(input_file, vocab_set))
    print(f" Fatto. Token unici processati: {len(freq)}")

    # C. Salvataggio Frequenze
    freq_out = f"vocab_freq_snap{n}.csv"
    with open(freq_out, "w", encoding="utf-8") as f:
        f.write("lemma,count\n") # Aggiunto header per comodità
        for w, c in freq.most_common():
            f.write(f"{w},{c}\n")

    # D. Calcolo Soglia (Percentile)
    if not freq:
        print("Attenzione: Nessun token trovato.")
        continue

    counts_array = np.array(list(freq.values()))
    soglia_10perc = np.percentile(counts_array, 10) # Usa np.percentile standard

    # E. Filtraggio e Export Vettori
    # Filtriamo tenendo solo le parole sopra la soglia
    vocab_frequente = {w for w, c in freq.items() if c > soglia_10perc}
    
    output_vec = f"fasttext_snap{n}_filt.vec"
    dim = model.get_dimension()

    print(f"Esportazione vettori filtrati (> {soglia_10perc:.2f})...", end="")
    with open(output_vec, "w", encoding="utf-8") as f:
        # Header formato .vec standard: "NUM_PAROLE DIMENSIONE"
        f.write(f"{len(vocab_frequente)} {dim}\n")
        
        for w in vocab_frequente:
            vec = model.get_word_vector(w)
            # Formattazione stringa efficiente
            vec_str = " ".join(f"{x:.6f}" for x in vec)
            f.write(f"{w} {vec_str}\n")
            
    print(" Fatto.")
    
    print(f"Stats Snap {n}: Originale={len(vocab_set)} -> Finale={len(vocab_frequente)}")

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\marti\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!



=== SNAP 1 ===
Vocab filtrato: 62150
Vocab originale: 65157
Vocab filtrato: 62150
Soglia 10 percentile: 6.0
Vocab finale: 50128

=== SNAP 2 ===
Vocab filtrato: 59062
Vocab originale: 61849
Vocab filtrato: 59062
Soglia 10 percentile: 6.0
Vocab finale: 47828

=== SNAP 3 ===
Vocab filtrato: 79314
Vocab originale: 83334
Vocab filtrato: 79314
Soglia 10 percentile: 6.0
Vocab finale: 64670

=== SNAP 4 ===
Vocab filtrato: 79926
Vocab originale: 83682
Vocab filtrato: 79926
Soglia 10 percentile: 6.0
Vocab finale: 65248

=== SNAP 5 ===
Vocab filtrato: 94227
Vocab originale: 98019
Vocab filtrato: 94227
Soglia 10 percentile: 6.0
Vocab finale: 76600

=== SNAP 6 ===
Vocab filtrato: 103834
Vocab originale: 108050
Vocab filtrato: 103834
Soglia 10 percentile: 6.0
Vocab finale: 84416

=== SNAP 7 ===
Vocab filtrato: 80677
Vocab originale: 84007
Vocab filtrato: 80677
Soglia 10 percentile: 6.0
Vocab finale: 65212

=== SNAP 8 ===
Vocab filtrato: 72491
Vocab originale: 75397
Vocab filtrato: 72491
Soglia 10 p

In [ ]:
values = np.array(list(freq.values()))

percentili = [1, 5, 10, 25, 50, 75, 90, 95, 99]

for p in percentili:
    print(f"{p:>3}° percentile -> freq = {np.percentile(values, p):.2f}")


  1° percentile -> freq = 5.00
  5° percentile -> freq = 5.00
 10° percentile -> freq = 6.00
 25° percentile -> freq = 8.00
 50° percentile -> freq = 17.00
 75° percentile -> freq = 54.00
 90° percentile -> freq = 211.00
 95° percentile -> freq = 582.40
 99° percentile -> freq = 3896.68


Example

In [ ]:
import fasttext
import multiprocessing
from pathlib import Path
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import random
import matplotlib.pyplot as plt


vec_file = "fasttext_snap1_filt.vec"

# dizionario parola -> vettore
word2vec = {}
with open(vec_file, encoding="utf-8") as f:
    header = f.readline()  # contiene num_words e dim, da ignorare
    for line in f:
        parts = line.rstrip().split()
        word = parts[0]
        vec = np.array([float(x) for x in parts[1:]], dtype=np.float32)
        word2vec[word] = vec


parola1 = "economico"
parola2 = "politico"

if parola1 in word2vec and parola2 in word2vec:
    vec1 = word2vec[parola1].reshape(1, -1)
    vec2 = word2vec[parola2].reshape(1, -1)

    sim = cosine_similarity(vec1, vec2)[0][0]
    distanza = 1 - sim

    print(f"Similarità tra '{parola1}' e '{parola2}': {sim:.4f}")
    print(f"Distanza coseno: {distanza:.4f}")
else:
    print(f"Una delle due parole non è presente: {parola1}, {parola2}")
